In [1]:
# Initial Environment Setup
try:
    from google.colab import drive
    COLAB_ENV = True
    print("Running in Google Colab environment.")
    drive.mount('/content/drive')
except ImportError:
    COLAB_ENV = False
    print("Not running in Google Colab environment.")

Not running in Google Colab environment.


In [2]:
# Core Imports
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import os
from datetime import datetime, timedelta
import time
import io
import ssl

In [3]:
# Google API Import
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

/opt/hostedtoolcache/Python/3.10.20/x64/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [4]:
# Data Cleaning & Processing
# Columns to exclude or drop to keep the BigQuery table efficient
COLUMNS_TO_DELETE = [
    'sequence_number', 'lot', 'bin', 'block', 'applicant_license_',
    'applicant_business_address', 'filing_representative_first_name',
    'filing_representative_last_name', 'owner_street_address',
    'owner_city', 'owner_state', 'owner_zip_code'
]

In [5]:
# Configuration Constant
Buid_Approved_Permit_URL = "https://data.cityofnewyork.us/resource/rbx6-tga4.csv"
API_LIMIT_PER_REQUEST = 1000
ZIP_CODES_TO_INCLUDE = [10004, 10005, 10006, 10007, 10038, 10280, 10282, 10013, 10002]
DEFAULT_INITIAL_FETCH_DATE = "2018-07-01"

In [6]:
# New BigQuery Configuration
GOOGLE_CLOUD_PROJECT_ID = "stable-liberty-426016-d2"
BIGQUERY_TABLE_ID = "nyc_dob_build_approved_permit.dob_build_permit"

# In this dataset, Manhattan CB1 is represented as 101
COMMUNITY_BOARD_CODE = 101
ZIP_CODES_TO_INCLUDE = [10004, 10005, 10006, 10007, 10038, 10280, 10282, 10013, 10002]
DEFAULT_INITIAL_FETCH_DATE = "2018-07-01"

# Dynamic path for the service account key
if COLAB_ENV:
    SERVICE_ACCOUNT_KEY_FILE = "/content/drive/Shareddrives/311_Complaint_Data/service_account_key.json"
else:
    SERVICE_ACCOUNT_KEY_FILE = "service_account_key.json"

In [7]:
# --- Helper Functions ---
def process_and_clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Cleans the raw construction permit data."""
    if df.empty: return df

    # 1. Drop irrelevant columns
    df.drop(columns=COLUMNS_TO_DELETE, inplace=True, errors='ignore')

    # 2. Convert date columns to datetime
    # DOB dataset uses 'issued_date', 'approved_date', and 'expired_date'
    date_cols = ['issued_date', 'approved_date', 'expired_date', 'filing_date']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            # Localize to NY time if applicable
            if df[col].dt.tz is not None:
                df[col] = df[col].dt.tz_convert('America/New_York').dt.tz_localize(None)

    # 3. Clean numeric fields (estimated costs)
    if 'estimated_job_costs' in df.columns:
        df['estimated_job_costs'] = pd.to_numeric(df['estimated_job_costs'], errors='coerce').fillna(0)

    return df

def fetch_nyc_data_incremental(start_date_str: str, end_date_str: str = None) -> pd.DataFrame:
    """Fetches construction data using the same chunking logic as 311 code."""
    all_fetched_dfs = []
    offset = 0
    more_data_available = True

    # Build the date and community board filter
    if end_date_str:
        date_filter = f"issued_date >= '{start_date_str}T00:00:00.000' AND issued_date <= '{end_date_str}T23:59:59.999'"
    else:
        date_filter = f"issued_date >= '{start_date_str}T00:00:00.000'"

    # CB1 Manhattan is coded as '101' in the DOB NOW dataset
    community_board_filter = "community_board = '101'"
    where_clause = f"{date_filter} AND {community_board_filter}"

    session = requests.Session()
    retries = Retry(total=10, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))

    while more_data_available:
        params = {'$limit': API_LIMIT_PER_REQUEST, '$offset': offset, '$where': where_clause, '$order': 'issued_date ASC'}
        try:
            response = session.get(Buid_Approved_Permit_URL, params=params, timeout=90)
            response.raise_for_status()
            if not response.text.strip(): more_data_available = False; continue

            page_df = pd.read_csv(io.StringIO(response.text))
            if not page_df.empty:
                all_fetched_dfs.append(page_df)
                offset += len(page_df)
                if len(page_df) < API_LIMIT_PER_REQUEST: more_data_available = False
            else:
                more_data_available = False
            time.sleep(1)
        except Exception as e:
            print(f"Error fetching chunk: {e}")
            break

    if not all_fetched_dfs: return pd.DataFrame()
    return pd.concat(all_fetched_dfs, ignore_index=True)

# <<< MODIFIED MAIN FUNCTION FOR BIGQUERY >>>
def update_bigquery_data():
    """Main function adjusted for Construction Data pipeline."""
    print("--- Starting DOB Build Permit BigQuery Update Process ---")

    try:
        creds = service_account.Credentials.from_service_account_file(
            SERVICE_ACCOUNT_KEY_FILE,
            scopes=["https://www.googleapis.com/auth/cloud-platform", "https://www.googleapis.com/auth/drive"]
        )
        print("Authentication successful.")
    except Exception as e:
        print(f"FATAL: Authentication failed: {e}"); return

    # Query BigQuery for the latest issued date
    last_processed_date_str = DEFAULT_INITIAL_FETCH_DATE
    try:
        print("Querying BigQuery for the last processed permit date...")
        sql = f"SELECT MAX(issued_date) as max_date FROM `{GOOGLE_CLOUD_PROJECT_ID}.{BIGQUERY_TABLE_ID}`"
        df_latest = pd.read_gbq(sql, project_id=GOOGLE_CLOUD_PROJECT_ID, credentials=creds)
        if not df_latest.empty and pd.notna(df_latest['max_date'][0]):
            last_processed_date = pd.to_datetime(df_latest['max_date'][0])
            last_processed_date_str = last_processed_date.strftime('%Y-%m-%d')
            print(f"Resuming from last date in BigQuery: {last_processed_date_str}")
    except Exception as e:
        print(f"Could not get last date from BigQuery: {e}. Using default start date.")

    # Calculate date range
    start_date = datetime.strptime(last_processed_date_str, '%Y-%m-%d') + timedelta(days=1)
    if start_date < datetime.strptime(DEFAULT_INITIAL_FETCH_DATE, '%Y-%m-%d'):
        start_date = datetime.strptime(DEFAULT_INITIAL_FETCH_DATE, '%Y-%m-%d')
    end_date = datetime.now()

    # Fetch in yearly chunks
    all_chunks_dfs = []
    for year in range(start_date.year, end_date.year + 1):
        chunk_start = max(start_date, datetime(year, 1, 1))
        chunk_end = min(end_date, datetime(year, 12, 31))
        if chunk_start > chunk_end: continue

        chunk_df = fetch_nyc_data_incremental(chunk_start.strftime('%Y-%m-%d'), chunk_end.strftime('%Y-%m-%d'))
        if not chunk_df.empty: all_chunks_dfs.append(chunk_df)

    if not all_chunks_dfs:
        print("\nNo new permits found to upload."); return

    new_data_df = pd.concat(all_chunks_dfs, ignore_index=True)

    # 1. Zip Code Filter
    new_data_df['zip_code'] = pd.to_numeric(new_data_df['zip_code'], errors='coerce')
    filtered_df = new_data_df[new_data_df['zip_code'].isin(ZIP_CODES_TO_INCLUDE)].copy()

    if filtered_df.empty:
        print("No data remains after zip filtering. Process finished."); return

    # 2. Process and Clean
    processed_df = process_and_clean_data(filtered_df)

    # 3. De-duplicate on work_permit (unique key for DOB NOW)
    if 'work_permit' in processed_df.columns:
        processed_df.drop_duplicates(subset=['work_permit'], inplace=True, keep='last')

    # 4. Upload
    print(f"\nUploading {len(processed_df)} cleaned permit rows to BigQuery...")
    try:
        processed_df.to_gbq(
            destination_table=BIGQUERY_TABLE_ID,
            project_id=GOOGLE_CLOUD_PROJECT_ID,
            if_exists='append',
            credentials=creds
        )
        print("Successfully uploaded construction data to BigQuery.")
    except Exception as e:
        print(f"FATAL: Failed to upload data: {e}")

if __name__ == "__main__":
    update_bigquery_data()

/tmp/ipykernel_2367/1151562834.py:86: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df_latest = pd.read_gbq(sql, project_id=GOOGLE_CLOUD_PROJECT_ID, credentials=creds)


--- Starting DOB Build Permit BigQuery Update Process ---
Authentication successful.
Querying BigQuery for the last processed permit date...


Resuming from last date in BigQuery: 2026-03-18



Uploading 16 cleaned permit rows to BigQuery...


/tmp/ipykernel_2367/1151562834.py:132: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  processed_df.to_gbq(


Successfully uploaded construction data to BigQuery.
